# **S3 Bucket Access: Sample Notebook**

EarthRISE \
Written by: Micky Maganini \
Point of Contact: eric.anderson@nasa.gov \
Last edited: 2026/06/17

## Part 1: Introduction

#**Please read these instructions before proceeding**

Welcome to the S3 bucket access sample notebook. This notebook will show you how to access data from Amazon Web Services Simple Storage Service, also known as AWS S3. S3 allows users to store large amounts of data similar to a service like Google Drive or Dropbox, and allows users to retrieve data at will. The NASA Commercial Smallsat Data Acquisition Program (or CSDA) distributes data to authorized end users via S3 buckets.

This file is a Google Colaboratory Notebook that allows you to navigate the S3 bucket and programmatically download files from the S3 bucket. Google Colaboratory notebooks combine code with text files. This notebook will download imagery from an S3 bucket to your Google Drive.

### Instructions
1. Make a copy of this notebook by clicking File > Save a copy in drive. Please close the original file and work in your copy of this file. After closing the original version of this notebook, proceed to step 2 in the copy of the notebook.
2. If you are unfamiliar with Google Colaboratory, proceed to Part 2 provides a quick crash course on how to interact with Colab notebooks. If you are already familiar with Google Colaboratory, proceed to Part 3.
3. Continue sequentially through the notebook, running each cell and reading the descriptions of each.

### Alternative Approaches
1. If you are not interested in working with coding, images can be downloaded directly from the following link:

     *   https://docs.google.com/spreadsheets/d/1W0wgh09iJu1hAdb7cEX57t70cUPEZ6_55RJEQFhN0jA/edit?usp=sharing

     This spreadsheet has 2 sheets, which can be navigated between by clicking at the bottom:
      * Imagery Descriptions: This sheet contains a description of the location, date, and image ID
      * Optical_Imagery_Downloads: This sheet contains download links for each file in the optical AWS S3 bucket



2. If you would prefer to download imagery directly to your local computer rather than to Google Drive, you can run the code from Part 4 onwards in your local terminal (e.g. command prompt on Windows), but will need to install the AWS Command Line Interface as shown in [these instructions.](https://docs.aws.amazon.com/cli/latest/userguide/cli-chap-getting-started.html) You can run each line of code as is, just exclude the exclamation point (!) at the beginning of the command

## Part 2: Google Colaboratory crash course

This notebook is intended to function in a user-friendly way such that those without prior coding or Golgle Colaboratory experience. Should you wish to learn more about Google Colaboratory before working with this notebook, you can review an introduction [here](https://colab.research.google.com/drive/1TY4OJICx1qnUuZmzh4kjvyFUin3HVG6V?usp=sharing).

## Part 3: Install Dependencies

Running the code cells below will allow you to install the relevant packages needed for this notebook. The only code cell you will need to modify is the first cell (labeled "Modify this variable"). After that, every code cell should be able to be run "as is".

In [ ]:
# Modify this variable!
my_Gdrive_folder = 'drive/MyDrive/AK_GIO/' # Make sure this beings with "drive/MyDrive/" and ends in a slash.

#This code would download data to the folder titled akgio within the 'My Drive' section of Google Drive.

In [ ]:
# Install relevant packages
!pip install boto3
!pip install geemap
!pip install awscli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 21.7 MB/s eta 0:00:00
  Attempting uninstall: rsa
    Found existing installation: rsa 4.9.1
    Uninstalling rsa-4.9.1:
      Successfully uninstalled rsa-4.9.1
  Attempting uninstall: docutils
    Found existing installation: docutils 0.21.2
    Uninstalling docutils-0.21.2:
      Successfully uninstalled docutils-0.21.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but yo

Import relevant packages below

In [ ]:
from google.colab import drive
import os
import boto3
from botocore.client import Config
from botocore import UNSIGNED
import rasterio

After running the following code cell, you will be prompted to click through a series of popups. Click approve or continue to click through these popups and connect this notebook to your Google Drive.

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.chdir('drive/MyDrive')

In [ ]:
# Split the Google Drive folder at each slash, and select a  subset of the folders
gdrive_split = my_Gdrive_folder.split('/')[2:-1]
gdrive_split

# Create and/or choose the folders defined by the user.
for i in gdrive_split:
  stri = str(i)
  if os.path.isdir(stri) == True:      #if folder already exists, enter into it
    os.chdir(stri)
  else:                                # if folder doeesn't yet exist, create it, then enter it.
    os.mkdir(stri)
    os.chdir(stri)

## Part 4: Navigating the AWS buckets

There are two AWS buckets that contain CSDA imagery for the Typhoon Halong event. These buckets have the following URIs (Uniform Resource Identifier):



*  s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/
*   s3://csda-maxar-distribution-west/earthrise-halong-ak-sar/




Let's use the ls command to list what is located within the optical bucket. Notice how we use the --no-sign-request flag. This is to tell AWS that we do not need to sign in because this data is hosted in a public bucket.

In [ ]:
!aws s3 ls s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/ --no-sign-request

                           PRE 102001009BB98A00/
                           PRE 10200100E2C31400/
                           PRE 10200100F2620D00/
                           PRE 102001010369BC00/
                           PRE 102001010FA30900/
                           PRE 1020010112776000/
                           PRE 1020010114E15700/
                           PRE 1020010118222100/
                           PRE 10300100BE8BA600/
                           PRE 10300100BED6EE00/
                           PRE 10300100D4936900/
                           PRE 10300100D49C1000/
                           PRE 10300100F011B900/
                           PRE 1030010101433100/
                           PRE 10300101042F7F00/
                           PRE 103001010E404D00/
                           PRE 103001011173DB00/
                           PRE 103001011224D500/
                           PRE 1030010115077E00/
                           PRE 10300101152BC200/
                    

Look at the output above. We see a list of items, each of which has the letters "PRE" beforehand. PRE means prefix, and indicates that this is a folder. Let's see what is in the first folder, the folder called "102001009BB98A00". In order to do this, we can use the same command, but add the name of this folder to the end of our command (after the slash, but before the no-sign-request flag).

In this case, a slash represents a file or folder nested within another folder. Looking at the code below, we are asking AWS to show us what is inside the folder titled 102001009BB98A00, which is located within the folder titled "earthrise-halong-ak-optical", which is located within the folder titled "csda-maxar-distribution-west".

In [ ]:
!aws s3 ls s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/ --no-sign-request

                           PRE 200013031559_01/
2026-06-09 15:05:12       5677 200013031559_01.MAN
2026-06-09 15:06:31          0 200013031559_01_EOT.TXT
2026-06-09 15:06:31       2256 6995237179862517075-request.json
2026-06-09 15:06:31       7589 6995237179862517075_manifest.json


This time, we see we have one folder (as indicated by the PRE tag), and 4 files. These files have tags consisting of numbers (indicating the amount of storage they take up). Another way to tell between files and folders is that folders will end in a / and files will end in a . followed by several letters (e.g. .txt). Let's see what is inside this folder.

In [ ]:
!aws s3 ls s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/ --no-sign-request

                           PRE 200013031559_01_P001_PAN/
                           PRE GIS_FILES/
2026-06-09 15:05:12      41342 200013031559_01_LAYOUT.JPG
2026-06-09 15:06:30       5033 200013031559_01_README.TXT
2026-06-09 15:06:30       6445 200013031559_01_README.XML


We can see we have two folders, let's navigate into the first folder and list what's there.

In [ ]:
!aws s3 ls s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/ --no-sign-request

2026-06-09 15:05:12      24319 20AUG29011436-P3DS-200013031559_01_P001-BROWSE.JPG
2026-06-09 15:05:12       3250 20AUG29011436-P3DS-200013031559_01_P001.IMD
2026-06-09 15:05:12      26835 20AUG29011436-P3DS-200013031559_01_P001.TIL
2026-06-09 15:05:12      44637 20AUG29011436-P3DS-200013031559_01_P001.XML
2026-06-09 15:05:12        470 20AUG29011436-P3DS-200013031559_01_P001_README.TXT
2026-06-09 15:05:12  536875641 20AUG29011436-P3DS_R01C1-200013031559_01_P001.TIF
2026-06-09 15:05:17  536875641 20AUG29011436-P3DS_R01C2-200013031559_01_P001.TIF
2026-06-09 15:05:20  363362681 20AUG29011436-P3DS_R01C3-200013031559_01_P001.TIF
2026-06-09 15:05:15  536875641 20AUG29011436-P3DS_R02C1-200013031559_01_P001.TIF
2026-06-09 15:05:26  536875641 20AUG29011436-P3DS_R02C2-200013031559_01_P001.TIF
2026-06-09 15:05:21  341746457 20AUG29011436-P3DS_R02C3-200013031559_01_P001.TIF
2026-06-09 15:05:29  536875641 20AUG29011436-P3DS_R03C1-200013031559_01_P001.TIF
2026-06-09 15:05:23  536875641 20AUG29011436

We can see that we have a .JPG file and many TIF files. The first .tif file is called 20AUG29011436-P3DS_R01C1-200013031559_01_P001.TIF. Since the file starts with "20AUG29", we know that it was acquired on August 29, 2020.

Now, let's navigate into the first folder in the other folder (earthrise-halong-ak-sar).

In [ ]:
!aws s3 ls s3://csda-maxar-distribution-west/earthrise-halong-ak-sar/ --no-sign-request

                           PRE 2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC/
                           PRE 2025-10-13-10-29-58_CAPELLA-17_SP_VV_GEC/
                           PRE 2025-10-14-01-09-39_CAPELLA-17_SP_VV_GEC/
                           PRE 2025-10-14-01-11-20_CAPELLA-17_SP_VV_GEC/
                           PRE 2025-10-14-02-45-53_CAPELLA-17_SP_VV_GEC/
                           PRE 2025-10-14-06-41-49_UMBRA-07/
                           PRE 2025-10-14-06-43-27_UMBRA-07/
                           PRE 2025-10-14-08-22-54_UMBRA-08/
                           PRE 2025-10-14-08-25-20_UMBRA-10/
                           PRE 2025-10-14-08-25-36_UMBRA-09/
                           PRE 2025-10-14-08-26-12_UMBRA-09/
                           PRE 2025-10-17-06-36-27_UMBRA-07/
                           PRE 2025-10-25-10-14-20_CAPELLA-17_SP_VV_GEC/
                           PRE 20251013T021900_ICEYE_X37_GRD_SLF_951723025/
                           PRE 20251013T082215_ICEYE_X34_GR

In [ ]:
!aws s3 ls s3://csda-maxar-distribution-west/earthrise-halong-ak-sar/2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC/ --no-sign-request

2026-06-08 19:36:40       3142 2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC.json
2026-06-08 19:36:40  847832373 2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC.tif
2026-06-08 19:36:40       2818 2025-10-13-10-28-20_CAPELLA-17_SP_VV_SICD.json
2026-06-08 19:36:42 3158283514 2025-10-13-10-28-20_CAPELLA-17_SP_VV_SICD.ntf
2026-06-08 19:36:40       2828 2025-10-13-10-28-20_CAPELLA-17_SP_VV_SIDD.json
2026-06-08 19:36:47  829441454 2025-10-13-10-28-20_CAPELLA-17_SP_VV_SIDD.ntf
2026-06-08 19:36:40        868 6995213430568228566-request.json
2026-06-08 19:36:40        602 6995213430568228566_manifest.json
2026-06-08 19:36:40       9693 license.txt


## Part 5: Download a Single File

Now we will download a single file from a folder. This will download a folder to our current directory in Google Drive. Let's look at our current directory in Google Drive, using the pwd command, which stands for "Print Working Directory".

In [ ]:
pwd

'/content/drive/MyDrive/AK_GIO'

Note that there are several forward slashes in this file. In my case, I see "/content/drive/MyDrive/AK_GIO". A slash represents a file or folder being nested within another folder. In this case, or working directory is the AK_GIO folder, which is located in the MyDrive folder, which represents our entire Google Drive. This is similar to how AWS S3 represents folders with slashes, as explained in part 4.

We can see what's currently in our working directory using the ls command

In [ ]:
ls

 102001009BB98A00/        10300100D49C1000/
'102001009BB98A00 (1)'/   10300100F011B900/
 10200100E2C31400/        1030010101433100/
 10200100F2620D00/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC.json
 102001010369BC00/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC.tif
 102001010FA30900/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SICD.json
 1020010112776000/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SICD.ntf
 1020010114E15700/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SIDD.json
 1020010118222100/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SIDD.ntf
 10300100BE8BA600/        6995213430568228566_manifest.json
 10300100BED6EE00/        6995213430568228566-request.json
 10300100D4936900/        license.txt


if nothing is output, that means our current working directory is empty. Now let's download a file into our Google Drive folder. Let's grab the 2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC.tif file from the most recent folder we were looking at.

In [ ]:
!aws s3 cp \
  s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R01C1-200013031559_01_P001.TIF \
 . \
 --no-sign-request

download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R01C1-200013031559_01_P001.TIF to ./20AUG29011436-P3DS_R01C1-200013031559_01_P001.TIF


After this code runs, it will take some time before the file appears in your google drive. The first line using the cp command in s3, which stands for copy. The second line is the "path" to the folder in the S3 bucket that we would like to download. The third line contains a . which tells Python that we want to download the file to our current directory in Google Drive. The last line tells AWS that we do not need to sign in because we are accessing a public bucket.

Let's run the ls command to see what is in our current Google Drive folder.

In [ ]:
ls

 102001009BB98A00/        10300100F011B900/
'102001009BB98A00 (1)'/   1030010101433100/
 10200100E2C31400/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC.json
 10200100F2620D00/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_GEC.tif
 102001010369BC00/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SICD.json
 102001010FA30900/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SICD.ntf
 1020010112776000/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SIDD.json
 1020010114E15700/        2025-10-13-10-28-20_CAPELLA-17_SP_VV_SIDD.ntf
 1020010118222100/        20AUG29011436-P3DS_R01C1-200013031559_01_P001.TIF
 10300100BE8BA600/        6995213430568228566_manifest.json
 10300100BED6EE00/        6995213430568228566-request.json
 10300100D4936900/        license.txt
 10300100D49C1000/


As we can see, we successfully downloaded our file! You can now download this file from Google Drive to your local computer.

## Part 6: Download an entire folder.

But what if we want to download an entire folder with one command rather than just a single file? Let's navigate to a folder within the optical bucket, then download it.

In [ ]:
!aws s3 ls s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/ --no-sign-request

2026-06-09 15:05:12      24319 20AUG29011436-P3DS-200013031559_01_P001-BROWSE.JPG
2026-06-09 15:05:12       3250 20AUG29011436-P3DS-200013031559_01_P001.IMD
2026-06-09 15:05:12      26835 20AUG29011436-P3DS-200013031559_01_P001.TIL
2026-06-09 15:05:12      44637 20AUG29011436-P3DS-200013031559_01_P001.XML
2026-06-09 15:05:12        470 20AUG29011436-P3DS-200013031559_01_P001_README.TXT
2026-06-09 15:05:12  536875641 20AUG29011436-P3DS_R01C1-200013031559_01_P001.TIF
2026-06-09 15:05:17  536875641 20AUG29011436-P3DS_R01C2-200013031559_01_P001.TIF
2026-06-09 15:05:20  363362681 20AUG29011436-P3DS_R01C3-200013031559_01_P001.TIF
2026-06-09 15:05:15  536875641 20AUG29011436-P3DS_R02C1-200013031559_01_P001.TIF
2026-06-09 15:05:26  536875641 20AUG29011436-P3DS_R02C2-200013031559_01_P001.TIF
2026-06-09 15:05:21  341746457 20AUG29011436-P3DS_R02C3-200013031559_01_P001.TIF
2026-06-09 15:05:29  536875641 20AUG29011436-P3DS_R03C1-200013031559_01_P001.TIF
2026-06-09 15:05:23  536875641 20AUG29011436

Recall earlier we talked about tags (i.e. the third column here). Recall that PRE indicated a folder, but now we just see numbers. These numbers indicate the number of bytes each file takes up. It is very important to make sure you do not try to download the entire root folder, as this will take a lot of time and space.

In [ ]:
!aws s3 sync \
  s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/ \
  . \
  --no-sign-request

download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS-200013031559_01_P001.XML to ./20AUG29011436-P3DS-200013031559_01_P001.XML
download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS-200013031559_01_P001_README.TXT to ./20AUG29011436-P3DS-200013031559_01_P001_README.TXT
download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS-200013031559_01_P001.TIL to ./20AUG29011436-P3DS-200013031559_01_P001.TIL
download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS-200013031559_01_P001-BROWSE.JPG to ./20AUG29011436-P3DS-200013031559_01_P001-BROWSE.JPG
download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/10200100

# Part 7: Downloading the Entire Bucket

What if I want to download the entire bucket? Remember how forward slashes (/) represent a separation between folders? All we have to do is repeat the command above, but chop off the end part of the command.


The below two commands will download every item from the "earthrise-halong-ak-sar" bucket and the "earthrise-halong-ak-optical" bucket respectively.

#**WARNING**: Before executing these commands, understand that this will take a lot of time to execute and a lot of storage space on your Google Drive (especially the optical bucket).

In [ ]:
!aws s3 sync \
  s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/ \
  . \
  --no-sign-request

download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R04C2-200013031559_01_P001.TIF to 102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R04C2-200013031559_01_P001.TIF
download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R04C3-200013031559_01_P001.TIF to 102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R04C3-200013031559_01_P001.TIF
download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R04C1-200013031559_01_P001.TIF to 102001009BB98A00/200013031559_01/200013031559_01_P001_PAN/20AUG29011436-P3DS_R04C1-200013031559_01_P001.TIF
download: s3://csda-maxar-distribution-west/earthrise-halong-ak-optical/102001009BB98A00/200013031559_01/200013031559_01_P001_P